In [1]:
# =======================
# 0️⃣ Imports
# =======================
import os, shutil, datetime
import numpy as np
import matplotlib.pyplot as plt
import itertools
from pathlib import Path

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, RocCurveDisplay

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input

In [2]:
DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
DATA_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split70_30_aug4000"  # train & val
TEST_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
DATA_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split70_30_aug4000"  # train & val
TEST_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

OUT_DIR = f"{DRIVE_ROOT}/DL/Emotion_Densenet_70-30_Aug4000_Results"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

def fresh_copy(src, dst):
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst)

fresh_copy(os.path.join(DATA_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(DATA_DIR, "val"),   LOCAL_VAL)
fresh_copy(TEST_DIR, LOCAL_TEST)

Mounted at /content/drive


In [3]:
# 3️⃣ Parameters
# =======================
IMG_HEIGHT = 48
IMG_WIDTH  = 48
BATCH_SIZE = 32
NUM_CLASSES = 7
EPOCHS_TOP = 10
EPOCHS_FINE = 10
LR = 1e-3


In [4]:
# =======================
# 4️⃣ Data Generators (no augmentation)
# =======================
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
val_datagen   = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen  = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    LOCAL_TRAIN,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    LOCAL_VAL,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


Found 28000 images belonging to 7 classes.
Found 8616 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


In [5]:
# =======================
# 5️⃣ Build DenseNet121 model
# =======================
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH,3))
base_model.trainable = False  # Initially freeze

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
preds = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=preds)

model.compile(optimizer=Adam(LR),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
              metrics=['accuracy'])

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [6]:
# =======================
# 6️⃣ Callbacks
# =======================
checkpoint = ModelCheckpoint(os.path.join(OUT_DIR, "best_model.h5"), monitor='val_accuracy', save_best_only=True, mode='max')
earlystop  = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-6)

callbacks = [checkpoint, earlystop, reduce_lr]

In [7]:
# =======================
# 7️⃣ Train Top Layers
# =======================
history_top = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_TOP,
    callbacks=callbacks
)

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


873/875 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2425 - loss: 2.1976

875/875 ━━━━━━━━━━━━━━━━━━━━ 69s 49ms/step - accuracy: 0.2426 - loss: 2.1967 - val_accuracy: 0.3887 - val_loss: 1.6762 - learning_rate: 0.0010
Epoch 2/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - accuracy: 0.3481 - loss: 1.7562 - val_accuracy: 0.3692 - val_loss: 1.7004 - learning_rate: 0.0010
Epoch 3/10
872/875 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3530 - loss: 1.7535

875/875 ━━━━━━━━━━━━━━━━━━━━ 22s 26ms/step - accuracy: 0.3530 - loss: 1.7534 - val_accuracy: 0.3907 - val_loss: 1.6844 - learning_rate: 0.0010
Epoch 4/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - accuracy: 0.3562 - loss: 1.7430 - val_accuracy: 0.3662 - val_loss: 1.7024 - learning_rate: 0.0010
Epoch 5/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - accuracy: 0.3599 - loss: 1.7409 - val_accuracy: 0.3801 - val_loss: 1.6871 - learning_rate: 0.0010
Epoch 6/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - accuracy: 0.3632 - loss: 1.7331 - val_accuracy: 0.3756 - val_loss: 1.6968 - learning_rate: 0.0010
Epoch 7/10
874/875 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3700 - loss: 1.7118

875/875 ━━━━━━━━━━━━━━━━━━━━ 44s 27ms/step - accuracy: 0.3700 - loss: 1.7118 - val_accuracy: 0.3914 - val_loss: 1.6803 - learning_rate: 5.0000e-04
Epoch 8/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3901 - loss: 1.6916

875/875 ━━━━━━━━━━━━━━━━━━━━ 23s 27ms/step - accuracy: 0.3900 - loss: 1.6916 - val_accuracy: 0.3983 - val_loss: 1.6657 - learning_rate: 5.0000e-04
Epoch 9/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - accuracy: 0.3754 - loss: 1.7002 - val_accuracy: 0.3936 - val_loss: 1.6723 - learning_rate: 5.0000e-04
Epoch 10/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 20s 23ms/step - accuracy: 0.3784 - loss: 1.6951 - val_accuracy: 0.3923 - val_loss: 1.6727 - learning_rate: 5.0000e-04


In [8]:
# =======================
# 8️⃣ Fine-Tune Full Model
# =======================
base_model.trainable = True  # unfreeze

# lower LR for fine-tuning
model.compile(optimizer=Adam(LR/10),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
              metrics=['accuracy'])

history_fine = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FINE,
    callbacks=callbacks
)

Epoch 1/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.3013 - loss: 2.0835

875/875 ━━━━━━━━━━━━━━━━━━━━ 214s 79ms/step - accuracy: 0.3014 - loss: 2.0832 - val_accuracy: 0.4990 - val_loss: 1.4588 - learning_rate: 1.0000e-04
Epoch 2/10
874/875 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.4939 - loss: 1.5183

875/875 ━━━━━━━━━━━━━━━━━━━━ 52s 60ms/step - accuracy: 0.4939 - loss: 1.5182 - val_accuracy: 0.5482 - val_loss: 1.3565 - learning_rate: 1.0000e-04
Epoch 3/10
874/875 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5756 - loss: 1.3400

875/875 ━━━━━━━━━━━━━━━━━━━━ 50s 57ms/step - accuracy: 0.5756 - loss: 1.3400 - val_accuracy: 0.5831 - val_loss: 1.2961 - learning_rate: 1.0000e-04
Epoch 4/10
874/875 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.6531 - loss: 1.1855

875/875 ━━━━━━━━━━━━━━━━━━━━ 50s 57ms/step - accuracy: 0.6531 - loss: 1.1856 - val_accuracy: 0.5989 - val_loss: 1.2817 - learning_rate: 1.0000e-04
Epoch 5/10
874/875 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.7127 - loss: 1.0690

875/875 ━━━━━━━━━━━━━━━━━━━━ 50s 57ms/step - accuracy: 0.7126 - loss: 1.0690 - val_accuracy: 0.6007 - val_loss: 1.2829 - learning_rate: 1.0000e-04
Epoch 6/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 47s 54ms/step - accuracy: 0.7709 - loss: 0.9577 - val_accuracy: 0.5766 - val_loss: 1.3714 - learning_rate: 1.0000e-04
Epoch 7/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 44s 50ms/step - accuracy: 0.8279 - loss: 0.8445 - val_accuracy: 0.5893 - val_loss: 1.3990 - learning_rate: 1.0000e-04
Epoch 8/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 46s 53ms/step - accuracy: 0.8856 - loss: 0.7349 - val_accuracy: 0.5911 - val_loss: 1.4168 - learning_rate: 1.0000e-04
Epoch 9/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.9287 - loss: 0.6431

875/875 ━━━━━━━━━━━━━━━━━━━━ 48s 55ms/step - accuracy: 0.9287 - loss: 0.6431 - val_accuracy: 0.6229 - val_loss: 1.4129 - learning_rate: 5.0000e-05
Epoch 10/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 46s 53ms/step - accuracy: 0.9662 - loss: 0.5680 - val_accuracy: 0.6156 - val_loss: 1.4586 - learning_rate: 5.0000e-05


In [9]:
# =======================
# 9️⃣ Evaluate on Test Set
# =======================
test_steps = test_gen.samples // BATCH_SIZE + 1
test_loss, test_acc = model.evaluate(test_gen, steps=test_steps)
print(f"Test Accuracy: {test_acc:.4f}")

225/225 ━━━━━━━━━━━━━━━━━━━━ 33s 146ms/step - accuracy: 0.5760 - loss: 1.5470
Test Accuracy: 0.6154


In [10]:
# =======================
# 🔟 Predictions & Metrics
# =======================
y_true = test_gen.classes
y_pred_prob = model.predict(test_gen, steps=test_steps)
y_pred = np.argmax(y_pred_prob, axis=1)
class_labels = list(test_gen.class_indices.keys())

225/225 ━━━━━━━━━━━━━━━━━━━━ 46s 153ms/step


In [11]:
# Classification report
report = classification_report(y_true, y_pred, target_names=class_labels)
with open(os.path.join(OUT_DIR, "classification_report.txt"), "w") as f:
    f.write(report)
print("✅ Classification report saved.")

✅ Classification report saved.


In [12]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(len(class_labels))
plt.xticks(tick_marks, class_labels, rotation=45)
plt.yticks(tick_marks, class_labels)
thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, cm[i, j], horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"))
plt.close()
print("✅ Confusion matrix saved.")


✅ Confusion matrix saved.


In [13]:
# ROC curves
plt.figure(figsize=(10,8))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve((y_true==i).astype(int), y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_labels[i]} (AUC={roc_auc:.2f})")
plt.plot([0,1],[0,1],'k--')
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.savefig(os.path.join(OUT_DIR, "roc_curve.png"))
plt.close()
print("✅ ROC curve saved.")



✅ ROC curve saved.


In [14]:


# Accuracy & Loss curves
history = {**history_top.history, **history_fine.history}

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history_top.history['accuracy'] + history_fine.history['accuracy'], label='Train Acc')
plt.plot(history_top.history['val_accuracy'] + history_fine.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history_top.history['loss'] + history_fine.history['loss'], label='Train Loss')
plt.plot(history_top.history['val_loss'] + history_fine.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.savefig(os.path.join(OUT_DIR, "accuracy_loss_curve.png"))
plt.close()
print("✅ Accuracy & Loss curves saved.")

print(f"All results saved in: {OUT_DIR}")

✅ Accuracy & Loss curves saved.
All results saved in: /content/drive/MyDrive/Colab Notebooks/DL/Emotion_Densenet_70-30_Aug4000_Results
